In [1]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
%matplotlib inline

##### Функция

In [3]:
def active_base(month_sales, register_base):
    month_sales['Nummeration'] = month_sales.reset_index().index + 1 # новая колонка для уникальной нумерации
    month_sales_with_reg = month_sales.merge(register_base, left_on='Id ГКП', right_on='ClientGroupID', how = 'left') \
                        .drop_duplicates(subset='Nummeration') # джойним продажи с регистрациями по idГКП, удаляем дубликаты, 
                                                               # оставляя исходное количество строк, как в продажах
    # приводим колонки дат к нужным типам для дальнейшей работы:
    month_sales_with_reg['Дата УПД'] = pd.to_datetime(month_sales_with_reg['Дата УПД'], dayfirst=True).dt.date 
    month_sales_with_reg['Дата УПД'] = pd.to_datetime(month_sales_with_reg['Дата УПД'], errors='coerce') #здесь надо оставить дататтайм, чтобы находить мин и макс (корректно работает только эта схема)
    month_sales_with_reg['RegisterDate'] = pd.to_datetime(month_sales_with_reg['RegisterDate']).dt.date #здесь хватит просто даты
    
    # создаём колонки минимальных и максимальных дат для каждого клиента:
    month_sales_with_reg['min_Дата_УПД'] = month_sales_with_reg.groupby('ClientGroupID')['Дата УПД'].transform('min')
    month_sales_with_reg['min_Дата_УПД'] = month_sales_with_reg['min_Дата_УПД'].dt.date

    month_sales_with_reg['max_Дата_УПД'] = month_sales_with_reg.groupby('ClientGroupID')['Дата УПД'].transform('max')
    month_sales_with_reg['max_Дата_УПД'] = month_sales_with_reg['max_Дата_УПД'].dt.date

    # создаём нужные переменные:
    #today = datetime.now() # либо задаём дату ПН либо вставляем now, но только в понедельник 
    week_ago = pd.to_datetime(today - timedelta(days=8)).date() # ВС неделю назад
    two_week_ago = pd.to_datetime(today - timedelta(days=15)).date() # ВС две недели назад
    cur_registration = pd.to_datetime(today - timedelta(days=29)).date() # ВС 4 недели назад, регистрации после этого дня считаем новыми
    cur_levels = ['Премиум', 'Бизнес']
    cur_levels_P = ['Премиум']
    cur_levels_B = ['Бизнес'] 
  
    # Общие данные
    # АКБ в начале периода
    ACB_start = month_sales_with_reg.query("@two_week_ago < `Дата УПД` <= @week_ago").query("`Сервисный уровень` in @cur_levels")
    ACB_last_week_users = ACB_start['ClientGroupID'].unique().tolist() # переменная со списком покупателей прошлой недели
    
    # новые:
    month_sales_with_reg_new_user = month_sales_with_reg.query("min_Дата_УПД > @week_ago") \
                                                        .query("RegisterDate > @cur_registration") \
                                                        .query("`Сервисный уровень` in @cur_levels")
    new_users = month_sales_with_reg_new_user['ClientGroupID'].unique().tolist() # переменная со списком новых

    # отток
    month_sales_with_reg_less_users = month_sales_with_reg.query("@two_week_ago < max_Дата_УПД <= @week_ago") \
                                                          .query("`Сервисный уровень` in @cur_levels")

    # реактив
    month_sales_with_reg_reactive = month_sales_with_reg.query("ClientGroupID not in @ACB_last_week_users") \
                                                        .query("max_Дата_УПД > @week_ago") \
                                                        .query('ClientGroupID not in @new_users') \
                                                        .query("`Сервисный уровень` in @cur_levels") 

    # АКБ в конце периода:
    ACB_final = month_sales_with_reg.query("`Дата УПД` > @week_ago").query("`Сервисный уровень` in @cur_levels") 

    # создаём итоговые переменные с цифрами уникальных idГКП по категориям и в процентах:
    new_user_count = month_sales_with_reg_new_user['ClientGroupID'].nunique()
    churn_count = month_sales_with_reg_less_users['ClientGroupID'].nunique()
    reactivation_count = month_sales_with_reg_reactive['ClientGroupID'].nunique()
    ACB_start_count = ACB_start['ClientGroupID'].nunique()
    ACB_final_count = ACB_final['ClientGroupID'].nunique()

    new_user_percentage = round(new_user_count / ACB_start_count * 100, 2) if ACB_start_count > 0 else 0
    churn_percentage = round(churn_count / ACB_start_count * 100, 2) if ACB_start_count > 0 else 0
    reactivation_percentage = round(reactivation_count / ACB_start_count * 100, 2) if ACB_start_count > 0 else 0

    # Премиум
    # АКБ в начале периода
    ACB_start_P = month_sales_with_reg.query("@two_week_ago < `Дата УПД` <= @week_ago").query("`Сервисный уровень` in @cur_levels_P")
    ACB_last_week_users_P = ACB_start_P['ClientGroupID'].unique().tolist() # переменная со списком покупателей прошлой недели
    
    # новые:
    month_sales_with_reg_new_user_P = month_sales_with_reg.query("min_Дата_УПД > @week_ago") \
                                                        .query("RegisterDate > @cur_registration") \
                                                        .query("`Сервисный уровень` in @cur_levels_P")
    new_users_P = month_sales_with_reg_new_user_P['ClientGroupID'].unique().tolist() # переменная со списком новых

    # отток
    month_sales_with_reg_less_users_P = month_sales_with_reg.query("@two_week_ago < max_Дата_УПД <= @week_ago") \
                                                          .query("`Сервисный уровень` in @cur_levels_P")

    # реактив
    month_sales_with_reg_reactive_P = month_sales_with_reg.query("ClientGroupID not in @ACB_last_week_users") \
                                                        .query("max_Дата_УПД > @week_ago") \
                                                        .query('ClientGroupID not in @new_users') \
                                                        .query("`Сервисный уровень` in @cur_levels_P") 

    # АКБ в конце периода:
    ACB_final_P = month_sales_with_reg.query("`Дата УПД` > @week_ago").query("`Сервисный уровень` in @cur_levels_P") 

    # создаём итоговые переменные с цифрами уникальных idГКП по категориям и в процентах:
    new_user_count_P = month_sales_with_reg_new_user_P['ClientGroupID'].nunique()
    churn_count_P = month_sales_with_reg_less_users_P['ClientGroupID'].nunique()
    reactivation_count_P = month_sales_with_reg_reactive_P['ClientGroupID'].nunique()
    ACB_start_count_P = ACB_start_P['ClientGroupID'].nunique()
    ACB_final_count_P = ACB_final_P['ClientGroupID'].nunique()

    new_user_percentage_P = round(new_user_count_P / ACB_start_count_P * 100, 2) if ACB_start_count_P > 0 else 0
    churn_percentage_P = round(churn_count_P / ACB_start_count_P * 100, 2) if ACB_start_count_P > 0 else 0
    reactivation_percentage_P = round(reactivation_count_P / ACB_start_count_P * 100, 2) if ACB_start_count_P > 0 else 0

    # Бизнес
    # АКБ в начале периода
    ACB_start_B = month_sales_with_reg.query("@two_week_ago < `Дата УПД` <= @week_ago").query("`Сервисный уровень` in @cur_levels_B")
    ACB_last_week_users_B = ACB_start_B['ClientGroupID'].unique().tolist() # переменная со списком покупателей прошлой недели
    
    # новые:
    month_sales_with_reg_new_user_B = month_sales_with_reg.query("min_Дата_УПД > @week_ago") \
                                                        .query("RegisterDate > @cur_registration") \
                                                        .query("`Сервисный уровень` in @cur_levels_B")
    new_users_B = month_sales_with_reg_new_user_B['ClientGroupID'].unique().tolist() # переменная со списком новых

    # отток
    month_sales_with_reg_less_users_B = month_sales_with_reg.query("@two_week_ago < max_Дата_УПД <= @week_ago") \
                                                          .query("`Сервисный уровень` in @cur_levels_B")

    # реактив
    month_sales_with_reg_reactive_B = month_sales_with_reg.query("ClientGroupID not in @ACB_last_week_users") \
                                                        .query("max_Дата_УПД > @week_ago") \
                                                        .query('ClientGroupID not in @new_users') \
                                                        .query("`Сервисный уровень` in @cur_levels_B") 

    # АКБ в конце периода:
    ACB_final_B = month_sales_with_reg.query("`Дата УПД` > @week_ago").query("`Сервисный уровень` in @cur_levels_B") 

    # создаём итоговые переменные с цифрами уникальных idГКП по категориям и в процентах:
    new_user_count_B = month_sales_with_reg_new_user_B['ClientGroupID'].nunique()
    churn_count_B = month_sales_with_reg_less_users_B['ClientGroupID'].nunique()
    reactivation_count_B = month_sales_with_reg_reactive_B['ClientGroupID'].nunique()
    ACB_start_count_B = ACB_start_B['ClientGroupID'].nunique()
    ACB_final_count_B = ACB_final_B['ClientGroupID'].nunique()

    new_user_percentage_B = round(new_user_count_B / ACB_start_count_B * 100, 2) if ACB_start_count_B > 0 else 0
    churn_percentage_B = round(churn_count_B / ACB_start_count_B * 100, 2) if ACB_start_count_B > 0 else 0
    reactivation_percentage_B = round(reactivation_count_B / ACB_start_count_B * 100, 2) if ACB_start_count_B > 0 else 0

    #выводим данные   
    statistics = {
            "АКБ на начало периода": ACB_start_count,
            "Новые пользователи за период": new_user_count,
            "Отток за период": - churn_count,
            "Реактивация за период": reactivation_count,
            "АКБ на конец периода": ACB_final_count,
            "% новых пользователей": f"{new_user_percentage} %",
            "% оттока": f"{churn_percentage} %",
            "% реактивации": f"{reactivation_percentage} %"}

    statistics_P = {
            "АКБ на начало периода": ACB_start_count_P,
            "Новые пользователи за период": new_user_count_P,
            "Отток за период": - churn_count_P,
            "Реактивация за период": reactivation_count_P,
            "АКБ на конец периода": ACB_final_count_P,
            "% новых пользователей": f"{new_user_percentage_P} %",
            "% оттока": f"{churn_percentage_P} %",
            "% реактивации": f"{reactivation_percentage_P} %"}
    
    statistics_B = {
            "АКБ на начало периода": ACB_start_count_B,
            "Новые пользователи за период": new_user_count_B,
            "Отток за период": - churn_count_B,
            "Реактивация за период": reactivation_count_B,
            "АКБ на конец периода": ACB_final_count_B,
            "% новых пользователей": f"{new_user_percentage_B} %",
            "% оттока": f"{churn_percentage_B} %",
            "% реактивации": f"{reactivation_percentage_B} %"}
    
    #создаём датафреймы отдельно
    common = pd.DataFrame(statistics.items(), columns=['Показатель', 'Общие данные']).reset_index(drop=True) 
    premium = pd.DataFrame(statistics_P.items(), columns=['Показатель', 'Премиум']).reset_index(drop=True) 
    busines = pd.DataFrame(statistics_B.items(), columns=['Показатель', 'Бизнес']).reset_index(drop=True) 

    #совмещаем
    combined = pd.concat([common, premium, busines], axis=1)
    combined = combined.loc[:, ~combined.columns.duplicated()]

    return combined

##### Ввод данных, применение функции от нужных таблиц 

In [80]:
#продажи 4 недели до 19.01.25 
month_sales_19_01 = pd.read_csv(r'C:\Users\1\Desktop\Для скрипта\Продажи 23.12.24-19.01.25.csv', sep=";")
register_base_23_01 = pd.read_csv(r'C:\Users\1\Desktop\Для скрипта\_select_a_GBD_ID_a_CreatedDate_as_Дата_создания_NКлиента_k_Clien_202501231427.csv', sep=",")

C:\Users\1\AppData\Local\Temp\ipykernel_5816\1012446166.py:2: DtypeWarning: Columns (19,61) have mixed types. Specify dtype option on import or set low_memory=False.
  month_sales_19_01 = pd.read_csv(r'C:\Users\1\Desktop\Для скрипта\Продажи 23.12.24-19.01.25.csv', sep=";")


In [81]:
today = pd.to_datetime('2025-01-20 00:00:00').date() 
active_base(month_sales_19_01, register_base_23_01)

,Показатель,Общие данные,Премиум,Бизнес
0,АКБ на начало периода,1259,512,747
1,Новые пользователи за период,3,1,2
2,Отток за период,-213,-21,-192
3,Реактивация за период,804,88,716
4,АКБ на конец периода,1853,580,1273
5,% новых пользователей,0.24 %,0.2 %,0.27 %
6,% оттока,16.92 %,4.1 %,25.7 %
7,% реактивации,63.86 %,17.19 %,95.85 %


In [84]:
#продажи 4 недели до 26.01.25 
month_sales_27_01 = pd.read_csv(r'C:\Users\1\Desktop\Продажи 30.12.25-26.01.25.csv', sep=";")
register_base_27_01 = pd.read_csv(r'C:\Users\1\Desktop\regs_202501270929.csv', sep=",")

C:\Users\1\AppData\Local\Temp\ipykernel_5816\3476474025.py:2: DtypeWarning: Columns (19,61) have mixed types. Specify dtype option on import or set low_memory=False.
  month_sales_27_01 = pd.read_csv(r'C:\Users\1\Desktop\Продажи 30.12.25-26.01.25.csv', sep=";")


In [86]:
today = pd.to_datetime('2025-01-27 00:00:00').date()
active_base(month_sales_27_01, register_base_27_01)

,Показатель,Общие данные,Премиум,Бизнес
0,АКБ на начало периода,1856,583,1273
1,Новые пользователи за период,5,4,1
2,Отток за период,-476,-29,-447
3,Реактивация за период,415,30,385
4,АКБ на конец периода,1800,588,1212
5,% новых пользователей,0.27 %,0.69 %,0.08 %
6,% оттока,25.65 %,4.97 %,35.11 %
7,% реактивации,22.36 %,5.15 %,30.24 %


In [33]:
#продажи 4 недели до 09.01.25 
month_sales_10_02 = pd.read_csv(r'C:\Users\1\Desktop\Для отчёта 10.02\Продажи 13.01-09.02.2025.csv', sep=";")
register_base_10_02 = pd.read_csv(r'C:\Users\1\Desktop\Для отчёта 10.02\regs_20250210.csv', sep=",")

C:\Users\1\AppData\Local\Temp\ipykernel_2892\2147008113.py:2: DtypeWarning: Columns (19,61) have mixed types. Specify dtype option on import or set low_memory=False.
  month_sales_10_02 = pd.read_csv(r'C:\Users\1\Desktop\Для отчёта 10.02\Продажи 13.01-09.02.2025.csv', sep=";")


In [35]:
today = pd.to_datetime('2025-02-10 00:00:00').date()
active_base(month_sales_10_02, register_base_10_02)

,Показатель,Общие данные,Премиум,Бизнес
0,АКБ на начало периода,1810,586,1224
1,Новые пользователи за период,5,5,0
2,Отток за период,-414,-21,-393
3,Реактивация за период,432,27,405
4,АКБ на конец периода,1833,597,1236
5,% новых пользователей,0.28 %,0.85 %,0.0 %
6,% оттока,22.87 %,3.58 %,32.11 %
7,% реактивации,23.87 %,4.61 %,33.09 %


In [12]:
#продажи 4 недели до 16.02.25 
month_sales_17_02 = pd.read_csv(r'C:\Users\1\Desktop\Для отчёта 17.02\Продажи 20.01-16.02.2025.csv', sep=";")
register_base_17_02 = pd.read_csv(r'C:\Users\1\Desktop\Для отчёта 17.02\regs_20250217.csv', sep=",")

C:\Users\1\AppData\Local\Temp\ipykernel_14900\3129907266.py:2: DtypeWarning: Columns (19,61) have mixed types. Specify dtype option on import or set low_memory=False.
  month_sales_17_02 = pd.read_csv(r'C:\Users\1\Desktop\Для отчёта 17.02\Продажи 20.01-16.02.2025.csv', sep=";")


In [14]:
today = pd.to_datetime('2025-02-17 00:00:00').date()
active_base(month_sales_17_02, register_base_17_02)

,Показатель,Общие данные,Премиум,Бизнес
0,АКБ на начало периода,1825,590,1235
1,Новые пользователи за период,10,9,1
2,Отток за период,-453,-21,-432
3,Реактивация за период,423,18,405
4,АКБ на конец периода,1805,596,1209
5,% новых пользователей,0.55 %,1.53 %,0.08 %
6,% оттока,24.82 %,3.56 %,34.98 %
7,% реактивации,23.18 %,3.05 %,32.79 %
